# 🎬 Wan 2.2 — Image to Video · לאה (מנוע איכות, חינם)
מחברת שנייה של הרשת — לצד LTX (מהירות), Wan 2.2 הוא מנוע ה**איכות**: תנועה אנושית זורמת יותר, פריימרים יפים יותר.

**הגדרות:** זמן ריצה ← שנה סוג זמן ריצה ← T4 GPU ← הרצה. **לפני הכל:** ריצה ← הפעל הכל (Run all).

⏱ כל קליפ של 5 שניות: ~15–40 דקות על T4. עלות: 0.
בנהינת שימי פיתוח 16/10/2026, לפי סריקת vidverses (Wan = המודל הפתוח הטוב ברשימה שלהם).

In [ ]:
!pip -q install diffusers transformers accelerate safetensors sentencepiece peft einops
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'אין GPU! הפעל T4')

## הגדרות — תמונת התחלה + פרומפט
תמונת ברירת המחדל: קנון הגוף של לאה (הנעילה של סאן). אפשר להחליף URL לכל תמונה מהגלריה: https://aidoor.shimi.land/gallery/

In [ ]:
IMAGE_URL = 'https://aidoor.shimi.land/gallery/lea-bikini/lea_bikini_2_fullcanon_seed161007.png'

# טכניקת הסצנות שאומצה מ-vidverses (המשכיות + בימוי)
PROMPT = '''A beautiful young woman with long dark hair dancing gracefully in a modern loft 
at golden hour. Natural fluid body movement, hips swaying, hands flowing through 
the air, soft cinematic lighting, warm tones, photorealistic, smooth motion.'''

NEGATIVE = 'blurry, low quality, distorted face, deformed hands, extra limbs, 
watermark, text, jittery motion, flickering'
SEED = 161007
FRAMES = 81   # ~5 שניות ב-16fps
HEIGHT, WIDTH = 480, 832

## טעינת המנוע (פעם אחת, ~4 דקות)
Wan2.2 I2V 5B — הגרסה הפתוחה הקלה, מותאמת ל-16GB של T4: הטקסט-אנקודר וה-VAE יורדים לזיכרון ה-CPU בזמן שלא נחוצים.

In [ ]:
import torch
from diffusers import WanImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

pipe = WanImageToVideoPipeline.from_pretrained(
    'Wan-AI/Wan2.2-I2V-5B-Diffusers',
    torch_dtype=torch.float16,   # T4 בלי bf16 — fp16 עובד
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print('✔ המנוע נטען')

## רינדור (~15–40 דקות. תוכלו להתנתק — Colab ממשיך ברקע כל עוד החיבור פתוח)

In [ ]:
img = load_image(IMAGE_URL)
g = torch.Generator('cuda').manual_seed(SEED)
out = pipe(
    image=img, prompt=PROMPT, negative_prompt=NEGATIVE,
    height=HEIGHT, width=WIDTH, num_frames=FRAMES, guidance_scale=5.0, generator=g,
).frames[0]
export_to_video(out, 'wan_output.mp4', fps=16)
print('✔ נשמר: wan_output.mp4')
from IPython.display import HTML
from base64 import b64encode
import ipywidgets as widgets
mp4 = open('wan_output.mp4','rb').read()
widgets.Video(value=mp4, width=640, loop=True)  # תצוגה מיידית

## העלאה חינם לגלריה (בלי מפתחות)
העלאה אנונימית ל-catbox — הקישור חוזר לכאן. שימי פיתוח יאסוף אותו לגלריית הרשת.

In [ ]:
import requests
r = requests.post('https://catbox.moe/user/api.php',
    data={'reqtype':'fileupload'}, files={'fileToUpload': open('wan_output.mp4','rb')})
print('🔗 קישור להורדה:', r.text if r.ok else 'נכשל: '+r.text)
print('שלחו את הקישור לשימי פיתוח — ייכנס לגלריה.')